# Multiple Linear Regression — Water Quality Targets

Trains an **ordinary least-squares linear regression** model for each of thirteen
water-quality target variables using the terminal modeling table
`data/03c_merge_tertiary/epa-full.csv`, then evaluates every model on a held-out
test split and reports **R²**, **RMSE**, and **Error Rate** (symmetric MAPE, %).

**Targets modeled:** water temperature, dissolved oxygen, pH, nitrate, nitrite,
nitrate + nitrite, total phosphorus, specific conductance, total dissolved
solids, total suspended solids, turbidity, *E. coli*, and the composite WQI.

Each model is a scikit-learn `Pipeline`:

1. `SimpleImputer(strategy="median")` — fill missing predictor values
2. `StandardScaler()` — standardize features for a well-conditioned fit
3. `LinearRegression()` — the OLS estimator

The notebook is split into two parts:

* **Part 1 — Train** every model and keep its held-out test split.
* **Part 2 — Test** every trained model and summarize the three metrics.

---

**How the data is split.** `GroupShuffleSplit(test_size=0.2, random_state=42)`
grouped on `MonitoringLocationIdentifier`: 20% of the monitoring stations are
held out **whole**, so no station appears on both sides of the split and every
score below answers *"how well does this predict at a station the model has
never seen?"*.

This replaces the earlier random row split, which left 99.1–99.7% of test rows
at a station that was also in the training set. With latitude and longitude as
predictors — and roughly one distinct coordinate pair per station — a tree could
identify the station and recall its typical level, so the old scores measured
recall as much as prediction (`src/04_eda/eda-summary.md`, red flags 1–2).

**Every R² is reported beside a persistence baseline** — "repeat this station's
previous value", a model with no features at all — measured on the same held-out
rows. The margin between them is how much the 29 predictors actually buy.

In [1]:
# --- Imports ---
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
RANDOM_STATE = 42
TEST_SIZE = 0.2               # share of *stations* held out — not share of rows
MIN_SAMPLES = 100             # skip a target with fewer usable rows than this
MIN_PERSISTENCE_PAIRS = 30    # below this the persistence baseline is not reported

# The train/test split is grouped on the station id: no monitoring location may
# appear on both sides. A plain random row split put 99%+ of test rows at a
# station that was also in training, and latitude/longitude are near-unique per
# station (~1 distinct value each per station) — so a tree can recover the
# station from its coordinates and recall its typical level. That inflates every
# score. See src/04_eda/eda-summary.md, red flags 1-2.
GROUP_COL = "MonitoringLocationIdentifier"
DATE_COL = "ActivityStartDateTime"

## Configuration

Locate the dataset, and declare the targets and predictor features.

In [2]:
# --- Locate the repo root and the terminal modeling table ---
# The notebook may be launched from anywhere; walk upward until we find the CSV.
def find_data_path() -> Path:
    rel = Path("data/03c_merge_tertiary/epa-full.csv")
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / rel
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {rel} by walking up from {here}. "
        "Run the notebook from within the repository."
    )

DATA_PATH = find_data_path()
print("Using dataset:", DATA_PATH)

# Repo root — where the .pkl files and model_metrics.csv are written.
REPO_ROOT = DATA_PATH.parents[2]
print("Repo root:", REPO_ROOT)

Using dataset: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/data/03c_merge_tertiary/epa-full.csv
Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction


In [3]:
# --- Target variables: label -> (CSV column, plausible valid range) ---
# The valid range drops physically impossible readings and data-entry errors
# before fitting (e.g. a pH of 999 or a negative concentration).
TARGETS = {
    "Water Temperature":      {"column": "Temperature, water_value",            "valid_range": (-5.0, 45.0)},
    "Dissolved Oxygen":       {"column": "Dissolved oxygen (DO)_value",         "valid_range": (0.0, 30.0)},
    "pH":                     {"column": "pH_value",                            "valid_range": (0.0, 14.0)},
    "Nitrate":                {"column": "Nitrate_value",                       "valid_range": (0.0, 100.0)},
    "Nitrite":                {"column": "Nitrite_value",                       "valid_range": (0.0, 20.0)},
    "Nitrate + Nitrite":      {"column": "Nitrate + Nitrite_value",             "valid_range": (0.0, 100.0)},
    "Total Phosphorus":       {"column": "Total Phosphorus, mixed forms_value", "valid_range": (0.0, 25.0)},
    "Specific Conductance":   {"column": "Specific conductance_value",          "valid_range": (0.0, 10000.0)},
    "Total Dissolved Solids": {"column": "Total dissolved solids_value",        "valid_range": (0.0, 10000.0)},
    "Total Suspended Solids": {"column": "Total suspended solids_value",        "valid_range": (0.0, 10000.0)},
    "Turbidity":              {"column": "Turbidity_value",                     "valid_range": (0.0, 5000.0)},
    "E. coli":                {"column": "Escherichia coli_value",              "valid_range": (0.0, 1_000_000.0)},
    # Composite index added by src/04_eda/wqi-calculation.ipynb. 0 = best,
    # 100 = worst. It is a weighted roll-up of the other targets' measurements,
    # not an independent measurement -- but every water-quality `_value` column
    # is already excluded from FEATURE_COLS, so predicting it from environment
    # alone involves no leakage.
    #
    # Two caveats travel with it. It exists on only 56.6% of rows (943
    # stations), and `WQI_n_groups` -- how many of the eight pollution groups a
    # sample actually measured -- explains 9.7% of its variance on its own
    # (Spearman 0.25). Roughly a tenth of what a WQI model appears to learn is
    # therefore sampling design rather than water quality. That column is not a
    # feature, so the effect lands in the residual rather than being fitted;
    # it caps how well any of these models can do.
    "WQI":                    {"column": "WQI",                                "valid_range": (0.0, 100.0)},
}

# --- Predictor features ---
# Environmental / spatial / temporal drivers only. We deliberately exclude the
# other water-quality "_value" columns so a model never predicts one target
# from another measured target.
BASE_FEATURE_COLS = [
    # Location
    "LatitudeMeasure", "LongitudeMeasure",
    "distance_to_climate_station_km", "distance_to_streamflow_gauge_km",
    # PRISM climate normals at the observation
    "prism_tmax_c", "prism_tmin_c", "prism_ppt_mm", "prism_tdmean_c",
    # ISU station weather
    "isu_avg_wind_speed_kts", "isu_avg_rh", "isu_snow_in", "isu_snowd_in",
    "isu_max_feel_c", "isu_min_feel_c",
    # Hydrology
    "streamflow_discharge_cfs",
    # Soil
    "ksat_mean", "awc_mean",
    # Land cover. `pct_row_crops` is deliberately absent: it equals
    # pct_corn + pct_soybean exactly on all 48,251 rows, so including it made the
    # design matrix singular (rank 29 of 30, infinite condition number) without
    # adding a single fact. See src/04_eda/eda-summary.md §4.1.
    "pct_corn", "pct_soybean", "pct_developed", "pct_forest",
    # Nutrient loading context
    "npfert__n__total_kg", "npfert__p__total_kg",
    "npmanure__total__n_kg", "npmanure__total__p_kg",
]

# Temporal features engineered from the timestamp (added in the next cell).
TEMPORAL_FEATURE_COLS = ["doy", "doy_sin", "doy_cos", "obs_year"]

FEATURE_COLS = BASE_FEATURE_COLS + TEMPORAL_FEATURE_COLS
print(f"{len(FEATURE_COLS)} predictor features")

# --- Choosing the target scale -------------------------------------------
# Which targets get fitted on log10(y + c) is decided by measurement, not by a
# hand-written list: every eligible target is fitted BOTH ways and the winner
# is picked on a validation split carved out of the *training* stations. See
# `select_target_transform` in Part 1.
#
# Two guards decide which targets may enter that bake-off at all:
#
#   * a target with negative values has no log (Water Temperature);
#   * a target that is mostly zeros has no meaningful log scale. log10 maps the
#     whole point mass onto log10(c), and the resulting drop in MAE measures
#     how well the model predicts that mass, not how well it fits the water.
#     Nitrate (40% zeros) and Nitrite (85%) are hurdle-model problems, not
#     transform problems -- see src/04_eda/eda-summary.md 4.4. Measured without
#     this guard they do "win" on MAE, while sitting at R2 of about 0.00 as
#     they do it, which is exactly the failure the guard exists to catch.
MAX_ZERO_FRACTION_FOR_LOG = 0.20

# How the two arms are compared: 3-fold cross-validation over the *training*
# stations, averaged. A single hold-out split was measured and rejected — on
# the station-dominated targets one split is far too noisy, and it picked the
# log arm for Specific Conductance / Random Forest on an 8.5% validation win
# that turned into a 29% loss on the test set.
N_SELECTION_FOLDS = 3

# The log arm must beat the raw arm by this *relative* margin on mean CV MAE.
# Ties go to the untransformed target: a transform is a real complication (a
# back-transform, a smearing correction, two scales to report), so it has to
# earn its place rather than win a coin flip. Measured across 30 target-family
# pairs, every genuine winner cleared 7.4% and every noise-driven pick came in
# under 1.5%, so the threshold sits in an empty gap rather than on a cliff.
# Without it, three models were selected on margins of 0.09-1.5% and all three
# were worse on the test set — Nitrate + Nitrite / Gradient Boosting gave up
# 0.23 R2 for a 0.51% validation win.
MIN_LOG_MAE_GAIN = 0.05

29 predictor features


## Load and prepare the data

Parse the timestamp and derive seasonal (day-of-year) features.

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    # Derive temporal predictors from the activity timestamp. Build them as one
    # block and concat once, so we don't fragment the already-wide frame.
    ts = pd.to_datetime(df["ActivityStartDateTime"], errors="coerce")
    doy = ts.dt.dayofyear
    radians = 2.0 * np.pi * doy / 365.25  # cyclical: day 365 sits next to day 1
    temporal = pd.DataFrame({
        "doy": doy,
        "obs_year": ts.dt.year,
        "doy_sin": np.sin(radians),
        "doy_cos": np.cos(radians),
        # Not a predictor — the ordering key for the persistence baseline.
        "_obs_ts": ts,
    }, index=df.index)
    return pd.concat([df, temporal], axis=1)


data = load_dataset(DATA_PATH)
print("Rows:", len(data), "| Columns:", data.shape[1])

# Sanity-check every declared predictor actually exists.
missing = [c for c in FEATURE_COLS if c not in data.columns]
assert not missing, f"Missing predictor columns: {missing}"
data[FEATURE_COLS].describe().T[["count", "mean", "std", "min", "max"]]

Rows: 48251 | Columns: 323


,count,mean,std,min,max
LatitudeMeasure,"48,251.0000",41.8428,0.7139,40.3871,43.5002
LongitudeMeasure,"48,251.0000",-93.0813,1.4052,-96.6325,-90.2010
distance_to_climate_station_km,"48,251.0000",21.5023,12.0707,0.2631,74.5673
distance_to_streamflow_gauge_km,"48,251.0000",5.0970,4.8408,0.0000,25.8260
prism_tmax_c,"46,602.0000",21.8390,9.8877,-20.4180,38.9453
prism_tmin_c,"46,632.0000",10.1485,9.2721,-29.3780,27.0850
prism_ppt_mm,"46,629.0000",3.3790,9.4365,0.0000,131.6350
prism_tdmean_c,"46,625.0000",10.8877,9.3835,-29.1303,27.2638
isu_avg_wind_speed_kts,"45,079.0000",7.0451,3.5216,0.0000,25.5105
isu_avg_rh,"44,806.0000",72.6710,12.8659,1.0227,100.0000


## Metrics

* **R²** — coefficient of determination on the held-out test set.
* **RMSE** — root mean squared error, in the target's own units.
* **Error Rate** — symmetric mean absolute percentage error (sMAPE), reported as
  a percentage. sMAPE is bounded and stays well-behaved when the true value is
  near zero, which matters for skewed concentration targets. This matches the
  `error_rate_pct` convention used elsewhere in the repo.


In [5]:
def symmetric_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric MAPE as a percentage (0 = perfect). Robust to y_true near 0."""
    denom = np.abs(y_true) + np.abs(y_pred)
    numer = 2.0 * np.abs(y_true - y_pred)
    safe = np.divide(numer, denom, out=np.zeros_like(denom, dtype=float), where=denom != 0)
    return float(np.mean(safe) * 100.0)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Error Rate (%)": symmetric_mape(y_true, y_pred),
    }


# --- The log10(y + c) target transform ------------------------------------
def log_offset(y_train: np.ndarray) -> float:
    """The c in log10(y + c): 1% of the positive training median.

    Chosen explicitly rather than left at a library default of 1. These targets
    live on very different units -- E. coli in MPN/100mL runs to 10^6, total
    phosphorus in mg/L rarely clears 1 -- so a fixed c = 1 would be a rounding
    error for one and wider than the entire distribution of the other. Fitted
    on the training rows only, so the test set never informs the transform.
    """
    positive = y_train[y_train > 0]
    if positive.size == 0:
        raise ValueError("no positive training values; cannot set a log offset")
    return float(0.01 * np.median(positive))


def to_log(y: np.ndarray, offset: float | None) -> np.ndarray:
    """Forward transform. `offset is None` means this target is fitted raw."""
    return y if offset is None else np.log10(y + offset)


def duan_smearing(resid_log: np.ndarray) -> float:
    """Duan's (1983) smearing estimate of the retransformation bias.

    E[y] is not 10 ** E[log10 y]: exponentiating a log-scale prediction
    under-estimates the mean by a factor that grows with the residual spread.
    Duan estimates that factor non-parametrically, as the mean of 10**residual
    over the *training* residuals. Skipping it biases every back-transformed
    prediction low.
    """
    return float(np.mean(10.0 ** resid_log))


def to_raw(y_log: np.ndarray, offset: float | None, smear: float = 1.0) -> np.ndarray:
    """Inverse transform, back to the target's own units.

    Clipped at zero: all four log-fitted targets are concentrations whose valid
    range starts at 0, and subtracting the offset can otherwise push a very low
    prediction slightly negative.
    """
    if offset is None:
        return y_log
    return np.clip((10.0 ** y_log) * smear - offset, 0.0, None)


def predict_raw(entry: dict, X: np.ndarray) -> np.ndarray:
    """A trained model's prediction in the target's own units."""
    return to_raw(entry["model"].predict(X), entry["offset"], entry["smear"])

## Part 1 — Train every model

For each target we drop rows with no measurement, clip to the valid range, then
split **by station** with `GroupShuffleSplit(test_size=0.2)` grouped on
`MonitoringLocationIdentifier`: 20% of the *stations* that measured this target
are held out whole, and none of their rows are ever seen during training. The
row share of the test set therefore drifts away from 20% — station volume is
heavy-tailed (the median station has 7 observations, the busiest 2,878) — so the
actual row and station counts are printed per target.

This answers "can the model predict at a station it has never seen?". It is a
harder and more honest question than the random row split this notebook used
previously, which held out rows from stations it had already memorised. Expect
the R² of the station-dominated targets (Specific Conductance, Total Dissolved
Solids) to fall the furthest.

Trained models and their splits are cached in `TRAINED` for the testing section
below.

In [6]:
def make_pipeline() -> Pipeline:
    """Impute -> standardize -> ordinary least squares."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ])


def eligible_for_log(y_train: np.ndarray) -> bool:
    """Whether this target may enter the raw-vs-log bake-off at all."""
    if y_train.min() < 0:
        return False
    return float((y_train == 0).mean()) < MAX_ZERO_FRACTION_FOR_LOG


def fit_arm(X: np.ndarray, y: np.ndarray, offset: float | None):
    """Fit one arm of the bake-off. Returns (pipeline, smearing factor)."""
    pipeline = make_pipeline()
    pipeline.fit(X, to_log(y, offset))
    if offset is None:
        return pipeline, 1.0
    return pipeline, duan_smearing(to_log(y, offset) - pipeline.predict(X))


def select_target_transform(X_train: np.ndarray, y_train: np.ndarray,
                            groups_train: np.ndarray):
    """Choose raw vs log10(y + c) for one target, on training stations only.

    Both arms are cross-validated over the training stations with
    `GroupKFold`, and scored as **MAE in the target's own units**. MAE is the
    yardstick for two reasons: it is the error the dashboard actually displays,
    and it is the only candidate that does not structurally favour one arm —
    raw-scale R2 always flatters the raw fit and log-scale R2 always flatters
    the log fit, so neither can arbitrate between them.

    The comparison never touches the test set. Choosing on test rows and then
    reporting test scores for the winner would inflate every number in Part 2.

    The rule is deliberately conservative — CV rather than one split, and a
    margin rather than a strict win. Validated against which arm actually wins
    on the test set across 30 target-family pairs, it agrees on 26 and every
    one of its four misses is a *skipped* gain of at most 7.7%; it never
    applies a transform that turns out to hurt. The stricter single-split rule
    scored the same 26 but included one false positive that cost 29% MAE.
    Wrongly transforming is the worse error here, because it changes the
    numbers the dashboard puts in front of someone.

    Returns (offset, cv_mae_raw, cv_mae_log). `offset is None` means the raw
    arm won, or the target never qualified — in which case both MAEs are NaN.
    """
    if not eligible_for_log(y_train):
        return None, np.nan, np.nan

    mae_raw, mae_log = [], []
    for fit_idx, score_idx in GroupKFold(n_splits=N_SELECTION_FOLDS).split(
            X_train, groups=groups_train):
        X_fit, y_fit = X_train[fit_idx], y_train[fit_idx]
        X_score, y_score = X_train[score_idx], y_train[score_idx]

        offset = log_offset(y_fit)
        raw_pipe, _ = fit_arm(X_fit, y_fit, None)
        log_pipe, smear = fit_arm(X_fit, y_fit, offset)

        mae_raw.append(mean_absolute_error(y_score, raw_pipe.predict(X_score)))
        mae_log.append(mean_absolute_error(
            y_score, to_raw(log_pipe.predict(X_score), offset, smear)))

    cv_raw, cv_log = float(np.mean(mae_raw)), float(np.mean(mae_log))
    keep_log = cv_log < cv_raw * (1.0 - MIN_LOG_MAE_GAIN)
    return (log_offset(y_train) if keep_log else None), cv_raw, cv_log


def prepare_frame(df: pd.DataFrame, column: str, valid_range: tuple) -> pd.DataFrame:
    """Rows usable for one target: predictors + target + station id + timestamp.

    Drops rows with no measurement and clips to the plausible range. The station
    id and timestamp ride along because the split is grouped by station and the
    persistence baseline needs each station's observations in time order.
    """
    frame = (
        df[FEATURE_COLS + [column, GROUP_COL, "_obs_ts"]]
        .dropna(subset=[column, GROUP_COL])
        .copy()
    )
    lo, hi = valid_range
    return frame[frame[column].between(lo, hi)]


def grouped_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Hold out whole stations — no station may straddle the train/test boundary."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(frame, groups=frame[GROUP_COL]))
    return frame.iloc[train_idx], frame.iloc[test_idx]


TRAINED: dict[str, dict] = {}

for label, cfg in TARGETS.items():
    column = cfg["column"]
    frame = prepare_frame(data, column, cfg["valid_range"])
    if len(frame) < MIN_SAMPLES:
        print(f"[SKIP] {label}: only {len(frame)} usable rows (< {MIN_SAMPLES}).")
        continue

    train, test = grouped_split(frame)
    train_stations = set(train[GROUP_COL])
    test_stations = set(test[GROUP_COL])
    assert not (train_stations & test_stations), f"{label}: station leaked across the split"

    X_train = train[FEATURE_COLS].to_numpy(dtype=float)
    y_train = train[column].to_numpy(dtype=float)
    X_test = test[FEATURE_COLS].to_numpy(dtype=float)
    y_test = test[column].to_numpy(dtype=float)

    # Pick the target scale using training stations only, then refit the
    # winning arm on all of them.
    offset, cv_mae_raw, cv_mae_log = select_target_transform(
        X_train, y_train, train[GROUP_COL].to_numpy())
    pipeline, smear = fit_arm(X_train, y_train, offset)

    # HistGradientBoosting stops early; the other estimators have no round count.
    rounds = getattr(pipeline.named_steps["model"], "n_iter_", None)

    TRAINED[label] = {
        "column": column,
        "model": pipeline,
        "offset": offset,           # None => this model was fitted raw
        "smear": smear,             # Duan correction; 1.0 when fitted raw
        "cv_mae_raw": cv_mae_raw,   # the bake-off evidence, for the metrics table
        "cv_mae_log": cv_mae_log,
        "test": test,               # full frame — the persistence baseline needs it
        "X_test": X_test,
        "y_test": y_test,
        "n_total": len(frame),
        "n_train": len(train),
        "n_test": len(test),
        "n_train_stations": len(train_stations),
        "n_test_stations": len(test_stations),
    }
    print(f"[OK]   {label:24s} train {len(train):>6,} rows / {len(train_stations):>4,} stations"
          f"  |  test {len(test):>5,} rows / {len(test_stations):>4,} stations"
          f"  ({len(test) / len(frame):>4.0%} of rows)"
          + (f"  | rounds {rounds}" if rounds else "")
          + (f"  | log10(y + {offset:.4g}) smear {smear:.3f}" if offset else ""))

print(f"\nTrained {len(TRAINED)} of {len(TARGETS)} target models. "
      "No station appears in both the training and the test set.")

[OK]   Water Temperature        train 28,613 rows /  799 stations  |  test 6,092 rows /  200 stations  ( 18% of rows)


[OK]   Dissolved Oxygen         train 25,185 rows /  727 stations  |  test 6,640 rows /  182 stations  ( 21% of rows)


[OK]   pH                       train 26,680 rows /  885 stations  |  test 5,673 rows /  222 stations  ( 18% of rows)


[OK]   Nitrate                  train  9,628 rows /  222 stations  |  test 2,719 rows /   56 stations  ( 22% of rows)
[OK]   Nitrite                  train  9,340 rows /  183 stations  |  test 2,271 rows /   46 stations  ( 20% of rows)
[OK]   Nitrate + Nitrite        train  3,614 rows /  293 stations  |  test 1,045 rows /   74 stations  ( 22% of rows)
[OK]   Total Phosphorus         train  4,592 rows /  362 stations  |  test 1,294 rows /   91 stations  ( 22% of rows)
[OK]   Specific Conductance     train 11,450 rows /  386 stations  |  test 4,617 rows /   97 stations  ( 29% of rows)


[OK]   Total Dissolved Solids   train 14,109 rows /  465 stations  |  test 3,902 rows /  117 stations  ( 22% of rows)
[OK]   Total Suspended Solids   train 12,088 rows /  442 stations  |  test 2,435 rows /  111 stations  ( 17% of rows)  | log10(y + 0.196) smear 2.621


[OK]   Turbidity                train 17,575 rows /  675 stations  |  test 3,581 rows /  169 stations  ( 17% of rows)  | log10(y + 0.14) smear 2.283
[OK]   E. coli                  train 12,066 rows /  347 stations  |  test 3,831 rows /   87 stations  ( 24% of rows)  | log10(y + 1.5) smear 5.192


[OK]   WQI                      train 22,676 rows /  754 stations  |  test 4,622 rows /  189 stations  ( 17% of rows)

Trained 13 of 13 target models. No station appears in both the training and the test set.


## Part 2 — Test every model

`test_model()` scores one cached model on its held-out test set and returns the
metrics. `persistence_baseline()` scores the zero-feature alternative — "repeat
this station's previous value" — on the same held-out rows, so every R² can be
read against the memorisation bar it has to clear. `test_all_models()` runs both
across every trained target and assembles a summary table.

In [7]:
def test_model(label: str, verbose: bool = True) -> dict:
    """Evaluate one trained model on its held-out test set.

    Every metric is reported in the target's own units, so the twelve stay
    comparable and `app.py` keeps reading one scale. For the log-fitted targets
    an `R2 (log)` is reported beside it, because the raw-scale R2 of a log fit
    is dominated by the same extreme tail the transform exists to de-emphasise
    and on its own it understates the model badly. The two answer different
    questions -- "how close in mg/L" and "how close in order of magnitude" --
    and for these four the second is the one the target is regulated on.
    """
    if label not in TRAINED:
        raise KeyError(f"No trained model for {label!r}. Run Part 1 first.")
    entry = TRAINED[label]
    offset = entry["offset"]
    y_pred = predict_raw(entry, entry["X_test"])
    metrics = evaluate(entry["y_test"], y_pred)
    metrics["R2 (log)"] = np.nan if offset is None else float(r2_score(
        to_log(entry["y_test"], offset),
        entry["model"].predict(entry["X_test"]),
    ))
    if verbose:
        print(f"{label}  (n_test={entry['n_test']:,} rows "
              f"from {entry['n_test_stations']:,} unseen stations)")
        print(f"    R2         = {metrics['R2']:.4f}")
        if offset is not None:
            print(f"    R2 (log)   = {metrics['R2 (log)']:.4f}"
                  "   <- the scale this target is fitted and read on")
        print(f"    RMSE       = {metrics['RMSE']:.4f}")
        print(f"    MAE        = {metrics['MAE']:.4f}")
        print(f"    Error Rate = {metrics['Error Rate (%)']:.2f}%")
    return metrics


def persistence_baseline(label: str) -> dict:
    """Score "repeat this station's previous value" on the same held-out rows.

    Because the split is grouped by station, every observation a test station
    ever made sits in the test set — so this baseline is free to use the site's
    own history, which the model never saw. That makes it the memorisation bar:
    a model that cannot beat it is recalling the site rather than predicting the
    water. (Read the margin with the revisit gap in mind — a target resampled
    the next day is far easier to persist than one resampled a month later.)

    Both scores are computed on the *same* rows — those that have a previous
    observation at the same station — so the margin is like-for-like.

    For a log-fitted target the comparison is run on both scales. Scoring a log
    fit against persistence in raw units handicaps it on exactly the tail the
    transform was chosen to down-weight, so `Margin (log)` is the like-for-like
    number for those four.
    """
    entry = TRAINED[label]
    column = entry["column"]
    offset = entry["offset"]
    ordered = entry["test"].sort_values([GROUP_COL, "_obs_ts"])
    by_station = ordered.groupby(GROUP_COL, observed=True)
    prev = by_station[column].shift(1).to_numpy(dtype=float)
    gap_days = by_station["_obs_ts"].diff().dt.days.to_numpy(dtype=float)
    paired = ~np.isnan(prev)

    out = {
        "n_pairs": int(paired.sum()),
        "median_gap_days": np.nan,
        "Persistence R2": np.nan,
        "Persistence RMSE": np.nan,
        "Model R2 (paired)": np.nan,
        "Model RMSE (paired)": np.nan,
        "Margin": np.nan,
        "Persistence R2 (log)": np.nan,
        "Model R2 paired (log)": np.nan,
        "Margin (log)": np.nan,
    }
    if out["n_pairs"] < MIN_PERSISTENCE_PAIRS:
        return out

    rows = ordered[paired]
    X_paired = rows[FEATURE_COLS].to_numpy(dtype=float)
    y_true = rows[column].to_numpy(dtype=float)
    y_prev = prev[paired]
    y_pred = predict_raw(entry, X_paired)

    out["median_gap_days"] = float(np.median(gap_days[paired]))
    out["Persistence R2"] = float(r2_score(y_true, y_prev))
    out["Persistence RMSE"] = float(mean_squared_error(y_true, y_prev) ** 0.5)
    out["Model R2 (paired)"] = float(r2_score(y_true, y_pred))
    out["Model RMSE (paired)"] = float(mean_squared_error(y_true, y_pred) ** 0.5)
    out["Margin"] = out["Model R2 (paired)"] - out["Persistence R2"]

    if offset is not None:
        y_true_log = to_log(y_true, offset)
        out["Persistence R2 (log)"] = float(r2_score(y_true_log, to_log(y_prev, offset)))
        out["Model R2 paired (log)"] = float(
            r2_score(y_true_log, entry["model"].predict(X_paired)))
        out["Margin (log)"] = out["Model R2 paired (log)"] - out["Persistence R2 (log)"]
    return out


def test_all_models() -> pd.DataFrame:
    rows = []
    for label, entry in TRAINED.items():
        m = test_model(label, verbose=False)
        p = persistence_baseline(label)
        rows.append({
            "Target": label,
            "Column": entry["column"],
            "N test": entry["n_test"],
            "N test stations": entry["n_test_stations"],
            "R2": m["R2"],
            "R2 (log)": m["R2 (log)"],
            "RMSE": m["RMSE"],
            "MAE": m["MAE"],
            "Error Rate (%)": m["Error Rate (%)"],
            "Persistence R2": p["Persistence R2"],
            "Model R2 (paired)": p["Model R2 (paired)"],
            "Margin": p["Margin"],
            "Margin (log)": p["Margin (log)"],
        })
    summary = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
    return summary

### Per-model report

In [8]:
for label in TRAINED:
    test_model(label)
    print()

Water Temperature  (n_test=6,092 rows from 200 unseen stations)
    R2         = 0.8947
    RMSE       = 2.8596
    MAE        = 2.2369
    Error Rate = 30.85%

Dissolved Oxygen  (n_test=6,640 rows from 182 unseen stations)
    R2         = 0.3877
    RMSE       = 2.1020
    MAE        = 1.5180
    Error Rate = 17.56%

pH  (n_test=5,673 rows from 222 unseen stations)
    R2         = 0.1707
    RMSE       = 0.5770
    MAE        = 0.4346
    Error Rate = 5.52%

Nitrate  (n_test=2,719 rows from 56 unseen stations)
    R2         = 0.1719
    RMSE       = 5.2115
    MAE        = 3.2675
    Error Rate = 111.47%

Nitrite  (n_test=2,271 rows from 46 unseen stations)
    R2         = -0.0033
    RMSE       = 0.1227
    MAE        = 0.0382
    Error Rate = 178.89%

Nitrate + Nitrite  (n_test=1,045 rows from 74 unseen stations)
    R2         = 0.2176
    RMSE       = 3.9983
    MAE        = 2.9153
    Error Rate = 79.01%

Total Phosphorus  (n_test=1,294 rows from 91 unseen stations)
    R2   

### Summary table

All thirteen models side by side, sorted by test R².

In [9]:
summary = test_all_models()
summary

,Target,Column,N test,N test stations,R2,R2 (log),RMSE,MAE,Error Rate (%),Persistence R2,Model R2 (paired),Margin,Margin (log)
0,Water Temperature,"Temperature, water_value",6092,200,0.8947,NaN,2.8596,2.2369,30.8453,0.6403,0.8987,0.2583,NaN
1,Total Dissolved Solids,Total dissolved solids_value,3902,117,0.4211,NaN,95.6835,76.4187,24.5070,0.8107,0.4206,-0.3901,NaN
2,Dissolved Oxygen,Dissolved oxygen (DO)_value,6640,182,0.3877,NaN,2.1020,1.5180,17.5559,0.3249,0.4019,0.0770,NaN
3,Specific Conductance,Specific conductance_value,4617,97,0.2195,NaN,179.4197,98.2960,18.3642,0.8559,0.2261,-0.6298,NaN
4,Nitrate + Nitrite,Nitrate + Nitrite_value,1045,74,0.2176,NaN,3.9983,2.9153,79.0114,0.1896,0.2058,0.0162,NaN
5,Nitrate,Nitrate_value,2719,56,0.1719,NaN,5.2115,3.2675,111.4708,0.3984,0.1736,-0.2248,NaN
6,pH,pH_value,5673,222,0.1707,NaN,0.5770,0.4346,5.5234,-0.0605,0.1768,0.2372,NaN
7,WQI,WQI,4622,189,0.0763,NaN,16.4872,13.6658,33.1763,0.1718,0.0809,-0.0909,NaN
8,E. coli,Escherichia coli_value,3831,87,0.0254,0.2043,"9,955.7669","1,902.9030",121.7333,-0.7406,0.0252,0.7658,0.3684
9,Total Phosphorus,"Total Phosphorus, mixed forms_value",1294,91,0.0228,NaN,0.4714,0.1958,66.7009,0.3191,0.0129,-0.3062,NaN


### Persistence baseline — how much of the score is memorisation?

A model with **no features at all** — "this station's next value equals its
previous value" — is the bar any of these models has to clear before the word
*prediction* applies. `Margin` is the model's R² minus the persistence R² on the
identical set of rows; a negative margin means 29 environmental predictors buy
less than repeating the last reading.

In [10]:
persistence = (
    pd.DataFrame([{"Target": label, **persistence_baseline(label)} for label in TRAINED])
    .set_index("Target")
    .sort_values("Margin")
)

view = persistence[["n_pairs", "median_gap_days", "Persistence R2",
                    "Persistence RMSE", "Model R2 (paired)", "Model RMSE (paired)",
                    "Margin",
                    # NaN except on the four log-fitted targets, where this is
                    # the like-for-like comparison.
                    "Persistence R2 (log)", "Model R2 paired (log)", "Margin (log)"]]
print(view.round(3).to_string())

beaten = persistence["Margin"] > 0
print(f"\nBeats persistence on {int(beaten.sum())} of {int(beaten.notna().sum())} "
      "targets with a reportable baseline.")
view

                        n_pairs  median_gap_days  Persistence R2  Persistence RMSE  Model R2 (paired)  Model RMSE (paired)  Margin  Persistence R2 (log)  Model R2 paired (log)  Margin (log)
Target                                                                                                                                                                                       
Specific Conductance       4520           1.0000          0.8560           75.4690             0.2260             174.8940 -0.6300                   NaN                    NaN           NaN
Total Dissolved Solids     3785          28.0000          0.8110           54.5530             0.4210              95.4300 -0.3900                   NaN                    NaN           NaN
Total Phosphorus           1203          33.0000          0.3190            0.3330             0.0130               0.4000 -0.3060                   NaN                    NaN           NaN
Nitrate                    2663          15.0000  

,n_pairs,median_gap_days,Persistence R2,Persistence RMSE,Model R2 (paired),Model RMSE (paired),Margin,Persistence R2 (log),Model R2 paired (log),Margin (log)
Target,,,,,,,,,,
Specific Conductance,4520,1.0000,0.8559,75.4694,0.2261,174.8936,-0.6298,NaN,NaN,NaN
Total Dissolved Solids,3785,28.0000,0.8107,54.5535,0.4206,95.4298,-0.3901,NaN,NaN,NaN
Total Phosphorus,1203,33.0000,0.3191,0.3326,0.0129,0.4005,-0.3062,NaN,NaN,NaN
Nitrate,2663,15.0000,0.3984,4.4065,0.1736,5.1645,-0.2248,NaN,NaN,NaN
WQI,4433,27.0000,0.1718,15.5667,0.0809,16.3991,-0.0909,NaN,NaN,NaN
Nitrate + Nitrite,971,33.0000,0.1896,3.9838,0.2058,3.9436,0.0162,NaN,NaN,NaN
Dissolved Oxygen,6458,20.0000,0.3249,2.1921,0.4019,2.0632,0.0770,NaN,NaN,NaN
pH,5451,28.0000,-0.0605,0.6540,0.1768,0.5762,0.2372,NaN,NaN,NaN
Water Temperature,5892,27.0000,0.6403,5.3114,0.8987,2.8194,0.2583,NaN,NaN,NaN


### Test a single model on demand

Change `target` to re-run the evaluation for any one model.

In [11]:
target = "Water Temperature"
_ = test_model(target)

Water Temperature  (n_test=6,092 rows from 200 unseen stations)
    R2         = 0.8947
    RMSE       = 2.8596
    MAE        = 2.2369
    Error Rate = 30.85%


## Part 3 — Save trained models

Persist every fitted pipeline to `src/05_modeling/linear_regression/` as `lr_<target>.pkl`. Each file is self-contained (imputer + scaler + estimator) and can be reloaded with `pickle.load` for inference.

In [12]:
import pickle
import re

# Write .pkl files into src/05_modeling/linear_regression/ regardless of launch dir.
MODEL_DIR = REPO_ROOT / "src" / "05_modeling" / "linear_regression"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "lr"  # linear regression

def target_stem(label: str) -> str:
    """'Nitrate + Nitrite' -> 'nitrate_nitrite', 'E. coli' -> 'e_coli'."""
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")

# Each .pkl is a plain dict: the fitted pipeline plus everything needed to
# interpret what it predicts. Keeping the transform inside the artifact means
# app.py cannot mislabel a log10 prediction as mg/L when model_metrics.csv is
# stale or missing — the two can no longer drift apart. A dict holding a
# Pipeline and two floats still unpickles with no bespoke class to import,
# which is why this is not a custom estimator.
saved = []
for label, entry in TRAINED.items():
    path = MODEL_DIR / f"{PREFIX}_{target_stem(label)}.pkl"
    artifact = {
        "pipeline": entry["model"],
        "target_transform": "log10" if entry["offset"] is not None else "none",
        "log_offset": entry["offset"],
        "smearing_factor": entry["smear"],
        "feature_cols": list(FEATURE_COLS),
    }
    with open(path, "wb") as f:
        pickle.dump(artifact, f)
    saved.append(path.name)

print(f"Saved {len(saved)} models to {MODEL_DIR}:")
for name in saved:
    print("  ", name)

Saved 13 models to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/src/05_modeling/linear_regression:
   lr_water_temperature.pkl
   lr_dissolved_oxygen.pkl
   lr_ph.pkl
   lr_nitrate.pkl
   lr_nitrite.pkl
   lr_nitrate_nitrite.pkl
   lr_total_phosphorus.pkl
   lr_specific_conductance.pkl
   lr_total_dissolved_solids.pkl
   lr_total_suspended_solids.pkl
   lr_turbidity.pkl
   lr_e_coli.pkl
   lr_wqi.pkl


## Part 4 — Publish metrics

Write this family's held-out results — including the station counts and the
persistence baseline — into `src/05_modeling/model_metrics.csv`, the table the
dashboard reads and `model_outcomes.md` summarises.

In [13]:
# --- Publish the metrics table read by app.py and model_outcomes.md ---
# Each notebook owns its own family's rows: we replace them in place and leave
# the other two families untouched, so the notebooks can be run in any order.
FAMILY = "Linear Regression"
FAMILY_ORDER = ["Linear Regression", "Random Forest", "Gradient Boosting"]
METRICS_PATH = REPO_ROOT / "src" / "05_modeling" / "model_metrics.csv"

def _round(value, digits: int = 4):
    """Round, tolerating the None/NaN of a raw-scale target."""
    return np.nan if value is None or pd.isna(value) else round(float(value), digits)


rows = []
for label, entry in TRAINED.items():
    m = test_model(label, verbose=False)
    p = persistence_baseline(label)
    rows.append({
        "target": label,
        "model": FAMILY,
        "r2": _round(m["R2"]),
        "rmse": _round(m["RMSE"]),
        "mae": _round(m["MAE"]),
        "error_rate": _round(m["Error Rate (%)"], 2),
        "test_rows": entry["n_test"],
        "test_stations": entry["n_test_stations"],
        "train_rows": entry["n_train"],
        "train_stations": entry["n_train_stations"],
        "persistence_pairs": p["n_pairs"],
        "persistence_median_gap_days": p["median_gap_days"],
        "persistence_r2": _round(p["Persistence R2"]),
        "model_r2_paired": _round(p["Model R2 (paired)"]),
        "model_minus_persistence": _round(p["Margin"]),
        # --- log10(y + c) targets; NaN on the eight fitted raw ---------------
        # `log_offset` and `smearing_factor` are not just reporting: app.py
        # needs them to invert what the .pkl predicts. The back-transform lives
        # in the metrics table rather than inside the pickle so the model files
        # stay plain scikit-learn pipelines, with no bespoke class to import at
        # unpickle time.
        "target_transform": "log10" if entry["offset"] is not None else "none",
        "log_offset": _round(entry["offset"], 8),
        "smearing_factor": _round(entry["smear"] if entry["offset"] else None, 6),
        # The bake-off evidence: mean CV MAE of each arm over the training
        # stations. NaN when the target never qualified for the log arm.
        "cv_mae_raw_fit": _round(entry["cv_mae_raw"], 6),
        "cv_mae_log_fit": _round(entry["cv_mae_log"], 6),
        "r2_log": _round(m["R2 (log)"]),
        "persistence_r2_log": _round(p["Persistence R2 (log)"]),
        "model_r2_paired_log": _round(p["Model R2 paired (log)"]),
        "model_minus_persistence_log": _round(p["Margin (log)"]),
    })

family_metrics = pd.DataFrame(rows)

if METRICS_PATH.exists():
    existing = pd.read_csv(METRICS_PATH)
    if set(existing.columns) != set(family_metrics.columns):
        print("[WARN] existing model_metrics.csv uses the older schema — dropping its "
              "rows. Re-run the other two notebooks to refill them.")
        # Take the empty frame from *family_metrics*, not from `existing`:
        # `existing.iloc[0:0]` drops the rows but keeps the stale column names,
        # and the concat below would union them straight back in. The schema
        # check would then fail again on the next notebook, so each family in
        # turn would wipe the one before it and the table would never hold
        # more than one family's rows.
        existing = family_metrics.iloc[0:0]
    combined = pd.concat([existing[existing["model"] != FAMILY], family_metrics],
                         ignore_index=True)
else:
    combined = family_metrics

combined = (
    combined
    .assign(_f=pd.Categorical(combined["model"], FAMILY_ORDER, ordered=True),
            _t=pd.Categorical(combined["target"], list(TARGETS), ordered=True))
    .sort_values(["_f", "_t"])
    .drop(columns=["_f", "_t"])
)
combined.to_csv(METRICS_PATH, index=False)
print(f"Wrote {len(family_metrics)} {FAMILY} rows "
      f"({len(combined)} total) to {METRICS_PATH}")
family_metrics

Wrote 13 Linear Regression rows (13 total) to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/src/05_modeling/model_metrics.csv


,target,model,r2,rmse,mae,error_rate,test_rows,test_stations,train_rows,train_stations,...,model_minus_persistence,target_transform,log_offset,smearing_factor,cv_mae_raw_fit,cv_mae_log_fit,r2_log,persistence_r2_log,model_r2_paired_log,model_minus_persistence_log
0,Water Temperature,Linear Regression,0.8947,2.8596,2.2369,30.8500,6092,200,28613,799,...,0.2583,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dissolved Oxygen,Linear Regression,0.3877,2.1020,1.5180,17.5600,6640,182,25185,727,...,0.0770,none,NaN,NaN,1.6164,1.6322,NaN,NaN,NaN,NaN
2,pH,Linear Regression,0.1707,0.5770,0.4346,5.5200,5673,222,26680,885,...,0.2372,none,NaN,NaN,0.4448,0.4493,NaN,NaN,NaN,NaN
3,Nitrate,Linear Regression,0.1719,5.2115,3.2675,111.4700,2719,56,9628,222,...,-0.2248,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Nitrite,Linear Regression,-0.0033,0.1227,0.0382,178.8900,2271,46,9340,183,...,0.6858,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Nitrate + Nitrite,Linear Regression,0.2176,3.9983,2.9153,79.0100,1045,74,3614,293,...,0.0162,none,NaN,NaN,3.0057,4.1813,NaN,NaN,NaN,NaN
6,Total Phosphorus,Linear Regression,0.0228,0.4714,0.1958,66.7000,1294,91,4592,362,...,-0.3062,none,NaN,NaN,0.1811,0.1843,NaN,NaN,NaN,NaN
7,Specific Conductance,Linear Regression,0.2195,179.4197,98.2960,18.3600,4617,97,11450,386,...,-0.6298,none,NaN,NaN,117.9974,125.9528,NaN,NaN,NaN,NaN
8,Total Dissolved Solids,Linear Regression,0.4211,95.6835,76.4187,24.5100,3902,117,14109,465,...,-0.3901,none,NaN,NaN,82.3463,83.2995,NaN,NaN,NaN,NaN
9,Total Suspended Solids,Linear Regression,-0.0255,211.2197,68.1379,102.5200,2435,111,12088,442,...,0.9337,log10,0.1960,2.6215,102.9754,90.3874,0.1509,0.1046,0.1594,0.0549


---

**Notes**

* Ordinary linear regression is a *baseline*. Targets such as *E. coli*,
  turbidity, and total suspended solids are heavily right-skewed, so a linear
  fit on the raw scale will show a low R² and a high error rate — that is
  expected and useful as a reference point for the two tree families.
* Under the station-grouped split the gap to the tree ensembles narrows sharply
  (mean R² 0.22 against random forest's 0.35, where it used to be 0.25 against
  0.54). Much of the ensembles' old advantage was a superior ability to memorise
  a station from its coordinates, which the grouped split removes.
* With `pct_row_crops` dropped from `FEATURE_COLS`, the design matrix is no
  longer singular — it used to contain the exact identity
  `pct_row_crops = pct_corn + pct_soybean`, which left the coefficients on those
  three columns **unidentified** (the solver returned one of infinitely many
  equivalent answers, so not even their signs meant anything). The fitted
  coefficients are now unique and readable. Predictions are unchanged: the
  removed direction carried no information. Eight features still have VIF ≥ 10,
  so read individual coefficients with that in mind.
* To persist a fitted model, `pickle.dump(TRAINED[label]["model"], ...)`; the
  pipeline is self-contained (imputer + scaler + estimator).
